# 2. Features and Labels

WBS 2.4, 2.5 and 3.1. I build the seven features and the two labels for every ETF at every month-end, check them, set up the training windows and explore the 2010 to 2019 data.

In [ ]:
# Run first in every session: mount Drive, get the latest code, install libraries.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

%pip install --quiet -r {REPO_DIR}/requirements.txt

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, download, cleaning, features, windows, variables

for module in (config, download, cleaning, features, windows, variables):
    importlib.reload(module)

DIRS = config.data_dirs()
pd.set_option("display.width", 160)

## Load daily prices

In [ ]:
clean = cleaning.load_clean(DIRS["processed"])
raw_info = json.loads((DIRS["raw"] / config.FILES["raw_info"]).read_text())

print(f"{len(clean):,} rows, {clean['ticker'].nunique()} tickers, "
      f"{clean['date'].min():%Y-%m-%d} to {clean['date'].max():%Y-%m-%d}")
print("Raw file MD5:", raw_info["md5"])
cleaning.summarize(variables.check_table(clean, "Daily Prices").rename(columns={"column": "ticker"}))

## Daily features

In [ ]:
daily = features.daily_features(clean)
daily[daily["ticker"] == "SPY"].dropna().tail()

## Monthly data

One row per universe, ETF and month-end from January 2010. September 2026 has features but no labels yet.

In [ ]:
panel = features.build_panel(clean, daily)

print(panel.groupby(["universe", "period"]).size().unstack(fill_value=0))
print("Month-ends:", panel["date"].nunique())
print("Volume bias set to 0 on", int(panel["volume_filled"].sum()), "rows")
panel.head()

## Checks

Includes a look-ahead test: features at random month-ends are recomputed from prices cut off at that date and must match exactly. A FAIL stops the notebook.

In [ ]:
panel_checks = features.validate_panel(panel, clean)
column_checks = variables.check_table(panel, "Monthly Data")
print(panel_checks.to_string(index=False))
print()
print(column_checks.to_string(index=False))

all_checks = pd.concat([panel_checks.assign(column=""), column_checks], ignore_index=True)
if (all_checks["status"] == "FAIL").any():
    raise RuntimeError("A check failed. Do not use this table.")

## Training windows

For each of the 81 test months: the decision date, the training window and the 12-month validation slice. Growing window is the main design; 36 and 60 months are robustness checks.

In [ ]:
all_windows = {w: windows.training_windows(panel, w) for w in ["growing", *config.ROLLING_WINDOWS]}
window_table = pd.concat(all_windows.values(), ignore_index=True)

for w, table in all_windows.items():
    print(f"window {w}:")
    print(windows.check_windows(table, panel).to_string(index=False))
    print()
print(variables.check_table(window_table, "Training Windows")["status"].value_counts().to_dict())
all_windows["growing"].iloc[[0, 1, -1]]

## Save

In [ ]:
out = DIRS["processed"]
panel.to_parquet(out / config.FILES["monthly"], index=False)
panel.to_csv(out / "Monthly Data.csv", index=False)
window_table.to_parquet(out / config.FILES["windows"], index=False)
window_table.to_csv(out / "Training Windows.csv", index=False)
variables.to_frame().to_csv(out / config.FILES["variables"], index=False)
all_checks.to_csv(DIRS["reports"] / config.FILES["checks"], index=False)

info = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "raw_file_md5": raw_info["md5"],
    "rows": {u: int(n) for u, n in panel["universe"].value_counts().items()},
    "month_ends": int(panel["date"].nunique()),
    "features": config.FEATURES,
    "top_k": config.TOP_K,
    "test_months": [config.TEST_FIRST_MONTH, config.TEST_LAST_MONTH],
}
(out / config.FILES["monthly_info"]).write_text(json.dumps(info, indent=2))
print("Saved Monthly Data, Training Windows and Variables")

## Exploration (2010 to 2019, ETF_A)

Only pre-2020 rows, so the test period stays unseen until the main specification is locked (WBS 4.3).

In [ ]:
hist = panel[(panel["period"] == "history") & (panel["universe"] == "ETF_A")].copy()
print(f"{hist['date'].nunique()} months, {len(hist)} rows")

summary = hist[config.FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T
summary.to_csv(DIRS["reports"] / "Exploration Feature Summary.csv")
summary.round(4)

In [ ]:
# How often each ETF finishes in the top 3 (chance is 3/8 = 37.5%),
# and how often a top-3 ETF stays in the top 3 the next month.
top3_share = hist.groupby("ticker")["top3"].mean().sort_values(ascending=False)

hist = hist.sort_values(["ticker", "date"])
hist["top3_now"] = hist.groupby("ticker")["top3"].shift(1)
stay = hist.dropna(subset=["top3_now"]).groupby("top3_now")["top3"].mean()

print(top3_share.round(3).to_string())
print()
print(f"Top 3 next month if top 3 now:     {stay.get(1.0, np.nan):.3f}")
print(f"Top 3 next month if not top 3 now: {stay.get(0.0, np.nan):.3f}")
top3_share.rename("share of months in top 3").to_csv(DIRS["reports"] / "Exploration Top 3 Share.csv")

In [ ]:
# Rank correlation between each feature and next month's return across the
# 8 ETFs, computed every month and averaged. t-stat = mean / (std / sqrt(months)).
def monthly_ic(df, feature):
    return df.groupby("date").apply(
        lambda g: g[feature].corr(g["next_return"], method="spearman"),
        include_groups=False)

ic = pd.DataFrame({f: monthly_ic(hist, f) for f in config.FEATURES})
ic_table = pd.DataFrame({
    "mean correlation": ic.mean(),
    "std": ic.std(),
    "t-stat": ic.mean() / (ic.std() / np.sqrt(len(ic))),
    "share of months > 0": (ic > 0).mean(),
}).round(3)
ic_table.to_csv(DIRS["reports"] / "Exploration Feature Correlation.csv")
ic_table

In [ ]:
# How similar the features are to each other.
corr = hist[config.FEATURES].corr(method="spearman")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation between features (ETF_A, 2010 to 2019)")
plt.tight_layout()
plt.show()

In [ ]:
# Return gap between the true top 3 and the other 5: the most a perfect
# ranker could earn each month before fees.
gap = hist.groupby(["date", "top3"])["next_return"].mean().unstack()
gap = gap[1] - gap[0]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(gap.index, gap * 100, width=20, color="#3b7d74")
ax.axhline(gap.mean() * 100, color="black", linewidth=1, linestyle="--",
           label=f"average {gap.mean():.2%} per month")
ax.set_ylabel("top 3 minus rest (%)")
ax.set_title("Monthly return gap, true top 3 vs other 5 (2010 to 2019)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

In [ ]:
passed = int((all_checks["status"] == "PASS").sum())
failed = int((all_checks["status"] == "FAIL").sum())
print(f"Month-ends:   {panel['date'].nunique()} ({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
for u, n in panel["universe"].value_counts().sort_index().items():
    print(f"Rows {u}:   {n:,}")
print(f"Test months:  {len(all_windows['growing'])} ({config.TEST_FIRST_MONTH} to {config.TEST_LAST_MONTH})")
print(f"Checks:       {passed} passed, {failed} failed")